# The Curse of Dimensionality

**Topic:** Why High Dimensions Are Dangerous

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** why distance-based intuitions break down in high-dimensional spaces
- **Explain** why the amount of data needed grows exponentially with the number of features
- **Interpret** the convergence of pairwise distances as dimensions increase

> **Tip:** Set the slider to 2 dimensions and note the average distance. Slowly increase to 50. Notice that the average distance keeps growing but the ratio between the maximum and minimum distance converges toward 1 — in very high dimensions, every point is approximately the same distance from every other point.

---
## How we got here

In **10_feature_importance.ipynb** we saw that some features are not useful. The curse of dimensionality explains the mathematical reason why useless features are not just neutral — they actively hurt distance-based algorithms.

This connects directly to:
- **math/linear_algebra/06_norms_and_distances.ipynb** — you computed Euclidean and Manhattan distances and saw how norms measure separation

Here we show what happens to those distances as the number of dimensions explodes.

---
## Why this matters for data science

KNN, clustering algorithms, kernel methods, and anything else that relies on distance becomes unreliable in high dimensions. The distances stop being informative — every point looks equally far from every other point.

This is not a limitation of any specific algorithm. It is a geometric property of high-dimensional spaces. The practical response is dimensionality reduction (PCA, feature selection) before applying distance-based methods.

---
## Try it yourself

In [2]:
np.random.seed(42)

out_dist  = Output()
out_vol   = Output()

dim_slider = IntSlider(value=2, min=1, max=50, step=1,
    description="Dimensions:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"))

def avg_distance(dims, n_points=500):
    avgs = []
    for d in dims:
        points = np.random.uniform(0, 1, (n_points, d))
        diffs  = points[0] - points[1:]
        dists  = np.sqrt((diffs**2).sum(axis=1))
        avgs.append(dists.mean())
    return avgs

DIMS_ALL = list(range(1, 51))
avg_dists = avg_distance(DIMS_ALL)
data_needed = [10**(d/10) for d in DIMS_ALL]

def render(max_d):
    traces = [
        go.Scatter(x=DIMS_ALL[:max_d], y=avg_dists[:max_d], mode="lines+markers",
            line=dict(color=PALETTE["primary"], width=2.5),
            marker=dict(size=5), name="Avg distance"),
        go.Scatter(x=[max_d], y=[avg_dists[max_d-1]], mode="markers",
            marker=dict(size=12, color=PALETTE["secondary"], symbol="star"),
            name=f"d={max_d}: {avg_dists[max_d-1]:.3f}"),
    ]
    layout = base_layout(
        title=f"Average pairwise distance converges as dimensions grow (d={max_d})",
        xaxis_title="Number of dimensions", yaxis_title="Average distance between random points")
    with out_dist:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces, layout=layout)))

    traces2 = [
        go.Scatter(x=DIMS_ALL[:max_d], y=data_needed[:max_d], mode="lines",
            line=dict(color=PALETTE["secondary"], width=2.5),
            name="Data needed (relative)"),
    ]
    layout2 = base_layout(
        title="Data needed grows exponentially with dimensions",
        xaxis_title="Number of dimensions", yaxis_title="Relative data needed (log scale)")
    layout2.update(yaxis=dict(type="log"))
    with out_vol:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces2, layout=layout2)))

def on_change(change): render(dim_slider.value)
dim_slider.observe(on_change, names="value")
display(VBox([dim_slider, out_dist, out_vol]))
render(2)

---
## A second way to see it: shrinking neighborhoods

The chart above shows the curse through **distances** and as dimensions grow, the nearest and farthest points bunch together until "nearest neighbor" means almost nothing.

The widget below shows the *same curse from the opposite direction* through **volume**. Instead of asking "how far apart are points?", it asks "if I draw a box around part of the space, how much of my data does it catch?"

Here's the unintuitive part. A box covering **half of every axis** sounds like it should capture most of your data. In 1-D it catches 50%. But each new dimension multiplies in another 50% cut: 50% → 25% → 12.5% → … → `0.5^d`. By 10 dimensions, that "half-of-everything" box holds under 0.1% of the data. Drag the slider wider and the effect barely budges — even a box covering 80% of every axis collapses toward zero as dimensions climb.

Both views are saying one thing: **in high dimensions, "local" stops existing.** Distances flatten out, and any neighborhood small enough to be local is too small to contain data. That's why KNN, clustering, and kernel methods everything that assumes nearby points are meaningfully similar break down.

In [3]:
np.random.seed(42)

out_panels = Output()
out_curve  = Output()

edge_slider = FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05,
    description="Box edge (fraction of each axis):",
    style={"description_width": "210px"},
    readout_format=".2f",
    layout=widgets.Layout(width="520px"))

caption = widgets.HTML(layout=widgets.Layout(width="720px"))

# One fixed cloud of points in 3D; lower-D panels just use the first 1–2 columns.
N = 120
PTS3 = np.random.uniform(0, 1, (N, 3))

def captured_mask(pts, edge):
    """Which points fall inside the [0, edge] box on every shown axis."""
    return np.all(pts <= edge, axis=1)

def render(edge):
    # ---------- Panels: 1D, 2D, 3D ----------
    in1 = captured_mask(PTS3[:, :1], edge)
    in2 = captured_mask(PTS3[:, :2], edge)
    in3 = captured_mask(PTS3[:, :3], edge)
    IN, OUT = PALETTE["secondary"], PALETTE["muted"]

    # 1D — points on a line, red segment [0, edge]
    fig1 = go.Figure()
    fig1.add_shape(type="rect", x0=0, x1=edge, y0=-0.05, y1=0.05,
        fillcolor=PALETTE["secondary"], opacity=0.25, line_width=0)
    fig1.add_trace(go.Scatter(x=PTS3[in1, 0], y=np.zeros(in1.sum()),
        mode="markers", marker=dict(color=IN, size=7), showlegend=False))
    fig1.add_trace(go.Scatter(x=PTS3[~in1, 0], y=np.zeros((~in1).sum()),
        mode="markers", marker=dict(color=OUT, size=7, opacity=0.6), showlegend=False))
    l1 = base_layout(title=f"1-D: {in1.mean()*100:.0f}% captured",
        xaxis_title="x", yaxis_title="")
    l1.update(width=300, height=240, yaxis=dict(range=[-0.3, 0.3], showticklabels=False),
        xaxis=dict(range=[0, 1]))
    fig1.update_layout(l1)

    # 2D — points in a square, red box [0,edge]²
    fig2 = go.Figure()
    fig2.add_shape(type="rect", x0=0, x1=edge, y0=0, y1=edge,
        fillcolor=PALETTE["secondary"], opacity=0.25, line_width=0)
    fig2.add_trace(go.Scatter(x=PTS3[in2, 0], y=PTS3[in2, 1],
        mode="markers", marker=dict(color=IN, size=6), showlegend=False))
    fig2.add_trace(go.Scatter(x=PTS3[~in2, 0], y=PTS3[~in2, 1],
        mode="markers", marker=dict(color=OUT, size=6, opacity=0.6), showlegend=False))
    l2 = base_layout(title=f"2-D: {in2.mean()*100:.0f}% captured",
        xaxis_title="x", yaxis_title="y")
    l2.update(width=300, height=240, xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]))
    fig2.update_layout(l2)

    # 3D — points in a cube, red box [0,edge]³ drawn as a mesh
    fig3 = go.Figure()
    e = edge
    # 8 corners of the [0,e]^3 box
    cx = [0, e, e, 0, 0, e, e, 0]
    cy = [0, 0, e, e, 0, 0, e, e]
    cz = [0, 0, 0, 0, e, e, e, e]
    fig3.add_trace(go.Mesh3d(x=cx, y=cy, z=cz,
        i=[0,0,0,1,1,2,4,4,5,2,3,6], j=[1,2,4,2,5,3,5,7,6,6,7,7],
        k=[2,4,1,5,2,7,6,5,7,7,4,4],
        color=PALETTE["secondary"], opacity=0.25, showscale=False, hoverinfo="skip"))
    fig3.add_trace(go.Scatter3d(x=PTS3[in3, 0], y=PTS3[in3, 1], z=PTS3[in3, 2],
        mode="markers", marker=dict(color=IN, size=4), showlegend=False))
    fig3.add_trace(go.Scatter3d(x=PTS3[~in3, 0], y=PTS3[~in3, 1], z=PTS3[~in3, 2],
        mode="markers", marker=dict(color=OUT, size=3, opacity=0.5), showlegend=False))
    l3 = base_layout(title=f"3-D: {in3.mean()*100:.0f}% captured",
        xaxis_title="x", yaxis_title="y")
    l3.update(width=320, height=260, scene=dict(
        xaxis=dict(title="x", range=[0, 1]), yaxis=dict(title="y", range=[0, 1]),
        zaxis=dict(title="z", range=[0, 1]),
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.2))))
    fig3.update_layout(l3)

    with out_panels:
        clear_output(wait=True)
        display(HBox([go.FigureWidget(fig1), go.FigureWidget(fig2), go.FigureWidget(fig3)]))

    # ---------- Capture curve to high d ----------
    dims = np.arange(1, 21)
    main = edge ** dims
    refs = {0.3: PALETTE["accent"], 0.5: PALETTE["muted"], 0.8: PALETTE["primary"]}
    curve_traces = []
    for ev, col in refs.items():
        if abs(ev - edge) > 1e-9:
            curve_traces.append(go.Scatter(x=dims, y=ev**dims, mode="lines",
                line=dict(color=col, width=1.2, dash="dot"),
                name=f"edge={ev:.1f}", opacity=0.6))
    curve_traces.append(go.Scatter(x=dims, y=main, mode="lines+markers",
        line=dict(color=PALETTE["secondary"], width=3),
        marker=dict(size=5), name=f"edge={edge:.2f} (current)"))
    lc = base_layout(
        title=f"Fraction of data captured = (edge)^d  —  collapses exponentially",
        xaxis_title="Number of dimensions", yaxis_title="Fraction of data captured")
    lc.update(width=620, height=340, yaxis=dict(range=[0, 1]))
    with out_curve:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=curve_traces, layout=lc)))

    # ---------- Caption ----------
    pct = lambda d: (edge ** d) * 100
    caption.value = (
        f"<div style='font-size:13px; line-height:1.5; padding:6px 2px'>"
        f"A box covering <b>{edge*100:.0f}% of every axis</b> sounds like it should grab most "
        f"of your data. It captures {pct(1):.0f}% in 1-D, {pct(2):.0f}% in 2-D, "
        f"{pct(3):.0f}% in 3-D — and only {pct(10):.1f}% by 10-D. "
        f"<b>That's the curse as volume:</b> to capture a fixed fraction of the data, the box "
        f"has to grow toward covering the <i>entire</i> range on every axis — local "
        f"neighborhoods stop being local. The measured percentages on the panels wobble around "
        f"these ideal values only because we drew {N} random points.</div>")

def on_change(change): render(edge_slider.value)
edge_slider.observe(on_change, names="value")
display(VBox([edge_slider, out_panels, out_curve, caption]))
render(0.5)

---
## What's happening?

Finding a friend in a city (2D) is straightforward — north/south and east/west locate anyone. In a skyscraper (3D) it is harder — you need the floor too. Now imagine a city with 1,000 invisible spatial dimensions. Distance stops meaning anything useful.

The two widgets above show this from two angles:

**Distances flatten (widget 1).** As dimensions grow, *all* pairwise distances get larger — but the **min, mean, and max bunch together**, so the gap between your nearest and farthest neighbor shrinks *relative* to how far away everything is. The contrast ratio `(max − min) / mean` falls toward zero. Once nearest and farthest are nearly the same distance, "nearest neighbor" is barely distinguishable from "random point."

**Neighborhoods empty out (widget 2).** A box covering a fixed fraction of each axis captures `(edge)^d` of the data — an exponential collapse. To hold a fixed fraction of your data, the box has to grow until it nearly covers the entire range on every axis. There is no such thing as a small, local, populated neighborhood in high dimensions.

These are the same fact in two languages: **local structure disappears.** That is why every distance-based method suffers.

| Dimensions | Distance behavior | Neighborhood capture (½-axis box) | Algorithms affected |
|---|---|---|---|
| 2–3 | Nearest clearly closer than farthest | 12–25% of data | None — distance works fine |
| 10–50 | Contrast ratio shrinking | ~0.1% down to ~10⁻¹⁵ | KNN, Kernel SVM |
| 100+ | Near and far nearly identical | Effectively 0% | KNN, clustering, kernel methods |
| 1000+ | Distance meaningless | 0% | All distance-based |

**The practical response:** reduce dimensions before applying distance-based methods. PCA, feature selection, and learned embeddings all aim to collapse many weakly-informative dimensions into a few strongly-informative ones — restoring a space where "near" means something again.

---
## Real-world example: Gene expression data

A genomics dataset has 20,000 gene features per patient and 200 patients. Every pair of patients has similar gene expression profiles — not because they are biologically similar, but because in 20,000 dimensions everything converges.

Applying KNN directly gives results no better than random. PCA reduces the 20,000 features to 50 principal components that capture most of the variance. KNN on those 50 components finds genuine nearest neighbors and produces clinically meaningful clusters.

---
## Key takeaway

> **In high dimensions, distance stops being meaningful — every point is approximately equidistant from every other, which breaks any algorithm that relies on proximity to make decisions.**

---
*Next up: Model Complexity and Capacity — how much can a model learn, and what does that cost?*